# This notebook is ment to test the methodes of the Cell class and visualize it using synthetic data


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn

from tqdm import tqdm
from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook
from bokeh.palettes import Spectral10, Colorblind8
from concurrent.futures import ProcessPoolExecutor
from scipy.ndimage import gaussian_filter1d

# Import the cell analysis classes with reload capability
import importlib
import cell_analysis
importlib.reload(cell_analysis)
from cell_analysis import Cell, PopulationAnalyzer

print("Cell and PopulationAnalyzer classes imported successfully!")

output_notebook()
hv.extension('bokeh')

Cell and PopulationAnalyzer classes imported successfully!


Loading BokehJS ...

In [3]:
test_cell_id = 1
test_session = 'test_simple_spike_train'

df = pd.DataFrame.from_records([
# Trial 1: GO Left all ones
    {
        'cell_ID': test_cell_id,
        'cell_type': 'MSN',
        'trial_session': test_session,
        'type': 'GO',
        'dir': 0,
        'trial_failed': False,
        'go_cue': 200,
        'stop_cue': np.nan,
        'first_relevant_saccade': 260,
        'ssd_number': 1.0,
        'neural_data': np.cumsum(np.arange(1000)),
        'trial_number': 1,
    },
    # Trial 2: GO Right al zeros
    {
        'cell_ID': test_cell_id,
        'cell_type': 'MSN',
        'trial_session': test_session,
        'type': 'GO',
        'dir': 0,
        'trial_failed': False,
        'go_cue': 200,
        'stop_cue': np.nan,
        'first_relevant_saccade': 360,
        'ssd_number': 1.0,
        'neural_data': [],
        'trial_number': 22,
    },
    # Trial 3: GO Left, more spikes
    {
        'cell_ID': test_cell_id,
        'cell_type': 'MSN',
        'trial_session': test_session,
        'type': 'GO',
        'dir': 180,
        'trial_failed': False,
        'go_cue': 200,
        'stop_cue': np.nan,
        'first_relevant_saccade': 260,
        'ssd_number': 1.0,
        'neural_data': np.arange(1000),
        'trial_number': 3,
    },
    # Trial 4: GO right, spike at go cue
    {
        'cell_ID': test_cell_id,
        'cell_type': 'MSN',
        'trial_session': test_session,
        'type': 'GO',
        'dir': 180,
        'trial_failed': False,
        'go_cue': 200,
        'stop_cue': np.nan,
        'first_relevant_saccade': 260,
        'ssd_number': 1.0,
        'neural_data': [200],
        'trial_number': 400,
    }
])

test_cell = Cell(df)
test_cell.plot_raster(epok=(-200, 500)).opts(height=300)

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .HeatMap.I :HeatMap   [columns,index]   (value)

In [4]:
def create_synthetic_test_data():
    """
    Create synthetic test data for a single cell with controlled spike patterns.
    
    Test data specifications (all successful trials):
    1. GO trials:
       - go_cue at 200 ms
       - Left (dir=180): spikes at 220 and 240 ms
       - Right (dir=0): spikes at 320 and 340 ms
    
    2. STOP trials:
       - go_cue at 200 ms, stop_cue at 500 ms
       - Left (dir=180): spikes at 250 and 550 ms
       - Right (dir=0): spikes at 320 and 540 ms
    
    3. CONT trials:
       - go_cue at 200 ms, stop_cue at 500 ms
       - Left (dir=180): spikes at 250 and 550 ms
       - Right (dir=0): spikes at 320 and 540 ms
    """
    test_cell_id = 9999
    test_session = 'test_session'
    
    trials = [
        # Trial 1: GO Left
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 180,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': np.nan,
            'first_relevant_saccade': 260,
            'ssd_number': np.nan,
            'neural_data': [220, 240],
        },
        # Trial 2: GO Right
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': np.nan,
            'first_relevant_saccade': 360,
            'ssd_number': np.nan,
            'neural_data': [320, 340],
        },
        # Trial 3: STOP Left
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'STOP',
            'dir': 180,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': 500,
            'first_relevant_saccade': np.nan,
            'ssd_number': 1,
            'neural_data': [250, 550],
        },
        # Trial 4: STOP Right
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'STOP',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': 500,
            'first_relevant_saccade': np.nan,
            'ssd_number': 1,
            'neural_data': [320, 540],
        },
        # Trial 5: CONT Left
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'CONT',
            'dir': 180,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': 500,
            'first_relevant_saccade': 570,
            'ssd_number': 1,
            'neural_data': [250, 550],
        },
        # Trial 6: CONT Right
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'CONT',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': 500,
            'first_relevant_saccade': 560,
            'ssd_number': 1,
            'neural_data': [320, 540],
        },
    ]
    
    return pd.DataFrame(trials)

test_cell = Cell(create_synthetic_test_data())
test_cell.plot_raster(epok=(-200, 500)).opts(height=300)

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .HeatMap.I :HeatMap   [columns,index]   (value)

In [5]:
test_cell.data

,cell_ID,cell_type,trial_session,type,dir,trial_failed,go_cue,stop_cue,first_relevant_saccade,ssd_number,neural_data,spikes_aligned_to_go_cue
0,9999,MSN,test_session,CONT,0,False,200,500.0,560.0,1.0,"[320, 540]","[120.0, 340.0]"
1,9999,MSN,test_session,CONT,180,False,200,500.0,570.0,1.0,"[250, 550]","[50.0, 350.0]"
2,9999,MSN,test_session,GO,0,False,200,NaN,360.0,NaN,"[320, 340]","[120.0, 140.0]"
3,9999,MSN,test_session,GO,180,False,200,NaN,260.0,NaN,"[220, 240]","[20.0, 40.0]"
4,9999,MSN,test_session,STOP,0,False,200,500.0,NaN,1.0,"[320, 540]","[120.0, 340.0]"
5,9999,MSN,test_session,STOP,180,False,200,500.0,NaN,1.0,"[250, 550]","[50.0, 350.0]"


In [6]:
test_hist = test_cell.plot_histogram_by_type_direction(
    alignment_point='go_cue',
    bin_size=1,
    separate_ssd=False,
    normalize=False
)

# Display the plot
test_hist[180]['STOP']

:Overlay
   .Histogram.I :Histogram   [Time]   (Spike Count)
   .VLine.I     :VLine   [x,y]

In [7]:
trials = [{
    'cell_ID': test_cell_id,
    'cell_type': 'MSN',
    'trial_session': test_session,
    'type': 'CONT',
    'dir': 180,
    'trial_failed': False,
    'go_cue': 200,
    'stop_cue': 500,
    'first_relevant_saccade': 570,
    'ssd_number': 1,
    'neural_data': [250] if i % 2 == 0 else [250, 550],
} for i in range(10)]

test_cell = Cell(pd.DataFrame(trials))
test_hist = test_cell.plot_histogram_by_type_direction(
    alignment_point='go_cue',
    bin_size=1,
    separate_ssd=False,
    normalize=False
)

# Display the plot
test_hist[180]['CONT'].opts(
    opts.Histogram(line_width=5)
)

:Overlay
   .Histogram.I :Histogram   [Time]   (Spike Count)
   .VLine.I     :VLine   [x,y]

In [8]:
trials = [{
    'cell_ID': test_cell_id,
    'cell_type': 'MSN',
    'trial_session': test_session,
    'type': 'CONT',
    'dir': 180,
    'trial_failed': False,
    'go_cue': 200,
    'stop_cue': 500,
    'first_relevant_saccade': 570,
    'ssd_number': (i % 4) + 1,
    'neural_data': [250 + (i % 4) * 20],
} for i in range(20)]

test_cell = Cell(pd.DataFrame(trials))
test_hist = test_cell.plot_histogram_by_type_direction(
    alignment_point='go_cue',
    bin_size=1,
    separate_ssd=True,
    normalize=False
)

# Display the plot
test_hist[180]['CONT']
# print(test_hist[180]['CONT'])

:Overlay
   .NdOverlay.I                                                  :NdOverlay   [Element]
   .Histogram.SSD1_left_parenthesis_n_equals_5_right_parenthesis :Histogram   [Time]   (Spike Count)
   .Histogram.SSD2_left_parenthesis_n_equals_5_right_parenthesis :Histogram   [Time]   (Spike Count)
   .Histogram.SSD3_left_parenthesis_n_equals_5_right_parenthesis :Histogram   [Time]   (Spike Count)
   .Histogram.SSD4_left_parenthesis_n_equals_5_right_parenthesis :Histogram   [Time]   (Spike Count)
   .VLine.I                                                      :VLine   [x,y]